In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

from pathlib import Path
PROJECT   = Path('/content/drive/MyDrive/cm3070')
DATA      = PROJECT / 'data'
EXPENSIVE = PROJECT / 'expensive'
EXPENSIVE.mkdir(parents=True, exist_ok=True)

SUB    = 'stocks'
OUT    = EXPENSIVE / f'sentiment_finbert_{SUB}.parquet'
CK_DIR = EXPENSIVE / f'_finbert_parts_{SUB}'   # per-chunk checkpoint files
print('output ->', OUT)

Mounted at /content/drive
output -> /content/drive/MyDrive/cm3070/expensive/sentiment_finbert_stocks.parquet


In [2]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before running FinBERT.')
print('GPU:', torch.cuda.get_device_name(0))

GPU: Tesla T4


In [3]:
import pandas as pd, hashlib

def text_sha(t):
    return hashlib.sha256(t.encode('utf-8')).hexdigest()[:16]

d = pd.read_csv(DATA / f'{SUB}.csv')
d['id'] = d['id'].astype(str)
d['selftext'] = d['selftext'].fillna('')
d = d[(d['selftext'] != '[deleted]') & (d['author'] != '[deleted]')].copy()   # = load_posts()
d['full_text'] = (d['title'].fillna('') + ' ' + d['selftext']).astype(str).str.strip()
d['text_sha'] = d['full_text'].map(text_sha)

versions = d.groupby('id')['text_sha'].nunique()
if (versions > 1).any():
    raise ValueError(f'{int((versions > 1).sum())} post IDs have conflicting text; resolve before scoring.')
print(f'{int(d.duplicated("id").sum())} identical duplicate records dropped')
d = d.drop_duplicates('id')[['id', 'full_text', 'text_sha']]
n_empty = int((d['full_text'].str.len() == 0).sum())
d = d[d['full_text'].str.len() > 0].reset_index(drop=True)
print(f'{n_empty} empty posts excluded (merge as neutral 0.0) | {len(d)} to score')

0 identical duplicate records dropped
0 empty posts excluded (merge as neutral 0.0) | 66952 to score


In [4]:
%pip install -q transformers==5.17.0

import json, time, transformers
import numpy as np
from transformers import pipeline

MODEL = 'ProsusAI/finbert'
MODEL_REVISION = '4556d13015211d73dccd3fdd39d39232506f3e43'
MAX_TOKENS = 512
PREPROCESSING = 'title+" "+selftext; strip; [deleted] filtered; v2'
EXPECTED = {'positive', 'negative', 'neutral'}
SCORE_COLUMNS = ['id', 'text_sha', 'n_tokens', 'finbert_pos', 'finbert_neg', 'finbert_neu']

if transformers.__version__ != '5.17.0':
    raise RuntimeError('Restart the runtime after installing Transformers 5.17.0, then run all cells.')

# Recalculate paths from the configuration in Cell 1.
OUT = EXPENSIVE / f'sentiment_finbert_{SUB}.parquet'
MANIFEST_PATH = OUT.with_suffix('.manifest.json')
CK_DIR = EXPENSIVE / f'_finbert_parts_{SUB}'

clf = pipeline('text-classification', model=MODEL, revision=MODEL_REVISION,
               device=0, truncation=True, max_length=MAX_TOKENS,
               batch_size=32, top_k=None)
MANIFEST = {'model': MODEL, 'model_commit': MODEL_REVISION,
            'transformers': transformers.__version__, 'max_tokens': MAX_TOKENS,
            'preprocessing': PREPROCESSING}
print(MANIFEST)

def compatible(path):
    if not path.exists():
        return False
    stored = json.loads(path.read_text())
    return {key: stored.get(key) for key in MANIFEST} == MANIFEST

def write_json_atomic(value, path):
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)

def write_parquet_atomic(frame, path):
    temporary = path.with_name(path.name + '.tmp')
    frame.to_parquet(temporary, index=False)
    temporary.replace(path)

def validated_scores(frame):
    """Validate fresh or cached probabilities before they can be reused or exported."""
    if not set(SCORE_COLUMNS).issubset(frame.columns):
        raise ValueError('A FinBERT score table has missing columns.')
    out = frame[SCORE_COLUMNS].copy()
    if out[['id', 'text_sha']].isna().any().any():
        raise ValueError('FinBERT IDs and text hashes cannot be missing.')
    out['id'] = out.id.astype(str)
    probability_columns = ['finbert_pos', 'finbert_neg', 'finbert_neu']
    out[probability_columns] = out[probability_columns].apply(pd.to_numeric, errors='coerce')
    p = out[probability_columns].to_numpy(dtype=float)
    if (not np.isfinite(p).all() or ((p < 0) | (p > 1)).any()
            or not np.allclose(p.sum(axis=1), 1.0, rtol=0, atol=1e-3)):
        raise ValueError('FinBERT probabilities must be finite, in [0, 1], and sum to 1.')
    out['n_tokens'] = pd.to_numeric(out.n_tokens, errors='coerce')
    lengths = out.n_tokens.to_numpy(dtype=float)
    if not np.isfinite(lengths).all() or ((lengths < 0) | (lengths != np.floor(lengths))).any():
        raise ValueError('Token counts must be finite nonnegative integers.')
    return out

CK_DIR.mkdir(exist_ok=True)
ck_manifest = CK_DIR / 'manifest.json'
part_paths = sorted(CK_DIR.glob('part_*.parquet'))
if (ck_manifest.exists() and not compatible(ck_manifest)) or (part_paths and not ck_manifest.exists()):
    raise RuntimeError(f'{CK_DIR} contains incompatible or unverified checkpoints; move it aside first.')
write_json_atomic(MANIFEST, ck_manifest)
previous = [validated_scores(pd.read_parquet(path)) for path in part_paths]
if compatible(MANIFEST_PATH) and OUT.exists():
    previous.append(validated_scores(pd.read_parquet(OUT)))
done = {key for frame in previous for key in zip(frame.id, frame.text_sha)}
todo = d[[key not in done for key in zip(d.id, d.text_sha)]].reset_index(drop=True)
print(f'{len(d) - len(todo)} reusable, {len(todo)} to score')

# Use the highest suffix, so a gap in checkpoint numbering cannot overwrite a part.
part_n = max((int(path.stem.split('_')[-1]) for path in part_paths), default=-1) + 1
CHUNK, t0 = 2000, time.time()
for i in range(0, len(todo), CHUNK):
    part = todo.iloc[i:i+CHUNK]
    texts = part.full_text.tolist()
    n_tokens = [len(ids) for ids in clf.tokenizer(texts, truncation=False)['input_ids']]
    probabilities = []
    for result in clf(texts):
        p = {item['label'].lower(): float(item['score']) for item in result}
        if len(result) != 3 or set(p) != EXPECTED:
            raise ValueError('Unexpected FinBERT output labels.')
        probabilities.append(p)
    frame = validated_scores(pd.DataFrame({
        'id': part.id.values, 'text_sha': part.text_sha.values, 'n_tokens': n_tokens,
        'finbert_pos': [p['positive'] for p in probabilities],
        'finbert_neg': [p['negative'] for p in probabilities],
        'finbert_neu': [p['neutral'] for p in probabilities]}))
    write_parquet_atomic(frame, CK_DIR / f'part_{part_n:05d}.parquet')
    part_n += 1
    print(f'  {min(i+CHUNK, len(todo))}/{len(todo)}  {time.time()-t0:.0f}s')
print(f'done in {time.time()-t0:.0f}s')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 47.0 MB/s eta 0:00:00


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  438MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  438MB            

model.safetensors: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

{'model': 'ProsusAI/finbert', 'model_commit': '4556d13015211d73dccd3fdd39d39232506f3e43', 'transformers': '5.17.0', 'max_tokens': 512, 'preprocessing': 'title+" "+selftext; strip; [deleted] filtered; v2'}
66952 reusable, 0 to score
done in 0s


In [5]:
part_paths = sorted(CK_DIR.glob('part_*.parquet'))
if part_paths and not compatible(CK_DIR / 'manifest.json'):
    raise RuntimeError('Checkpoint manifest does not match the current scoring setup.')
frames = [validated_scores(pd.read_parquet(path)) for path in part_paths]
if compatible(MANIFEST_PATH) and OUT.exists():
    frames.append(validated_scores(pd.read_parquet(OUT)))
res = validated_scores(pd.concat(frames, ignore_index=True) if frames
                       else pd.DataFrame(columns=SCORE_COLUMNS))
current = set(zip(d.id, d.text_sha))
res = res.loc[np.array([key in current for key in zip(res.id, res.text_sha)], dtype=bool)]
res = res.drop_duplicates('id').sort_values('id').reset_index(drop=True)
if set(zip(res.id, res.text_sha)) != current:
    raise RuntimeError('Some current posts are unscored; rerun Cell 4.')

res['finbert_score'] = res.finbert_pos - res.finbert_neg
if len(res):
    res['finbert_label'] = (res[['finbert_pos', 'finbert_neg', 'finbert_neu']]
                            .idxmax(axis=1).str.replace('finbert_', '', regex=False))
else:
    res['finbert_label'] = pd.Series(dtype='object')
res['truncated'] = res.n_tokens > MAX_TOKENS
share_truncated = float(res.truncated.mean()) if len(res) else 0.0

write_parquet_atomic(res, OUT)
write_json_atomic({**MANIFEST, 'n_scored': len(res), 'n_empty': n_empty,
                   'share_truncated': share_truncated}, MANIFEST_PATH)
# Keep checkpoints until both final writes succeed. Remove only this job's known files.
for path in part_paths:
    path.unlink()
ck_manifest = CK_DIR / 'manifest.json'
if ck_manifest.exists():
    ck_manifest.unlink()
if CK_DIR.exists() and not any(CK_DIR.iterdir()):
    CK_DIR.rmdir()

print(f'saved {len(res)} scores → {OUT.name} | {share_truncated:.1%} truncated at {MAX_TOKENS} tokens')
print(f'{n_empty} empty posts will receive neutral scores in the main notebook')
print(res.finbert_label.value_counts().to_string())
print(f'mean score {res.finbert_score.mean():.4f}  sd {res.finbert_score.std():.4f}')

saved 66952 scores → sentiment_finbert_stocks.parquet | 5.1% truncated at 512 tokens
0 empty posts will receive neutral scores in the main notebook
finbert_label
neu    58196
neg     6293
pos     2463
mean score -0.0728  sd 0.2763
